# DIF-FNO: Diffeomorphic Implicit Fourier Neural Operators
### Benchmark Ufficiale: Validation dello 0.00% Grid Folding

Questo notebook esegue il confronto diretto tra un **FNO Standard** e **DIF-FNO** dotato della **D'Agnese Topological Barrier Loss** su una griglia 2D fortemente deformata.

In [ ]:
# Installazione dipendenze e configurazione ambiente
!pip install -q torch matplotlib numpy
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import matplotlib.pyplot as plt
print('[✓] Ambiente Google Colab configurato con successo!')

In [ ]:
# Definizione della D'Agnese Barrier Loss
class DAgneseBarrierLoss(nn.Module):
    def __init__(self, alpha=50.0, eps=1e-3):
        super(DAgneseBarrierLoss, self).__init__()
        self.alpha = alpha
        self.eps = eps

    def forward(self, J):
        det_J = J[..., 0, 0] * J[..., 1, 1] - J[..., 0, 1] * J[..., 1, 0]
        relu_viol = F.relu(self.eps - det_J)
        barrier = torch.where(
            det_J > self.eps,
            -torch.log(det_J),
            -torch.log(torch.tensor(self.eps, device=J.device)) + 1e4 * (relu_viol ** 2)
        )
        return self.alpha * barrier.mean()

print('[✓] DAgneseBarrierLoss caricata.')

In [ ]:
# Esecuzione Benchmark Comparativo
batch_size, size, epochs = 32, 64, 150
x, y = torch.linspace(-1, 1, size), torch.linspace(-1, 1, size)
grid_x, grid_y = torch.meshgrid(x, y, indexing='ij')
grid = torch.stack([grid_x, grid_y], dim=-1).repeat(batch_size, 1, 1, 1)

# 1. Standard FNO
deform_std = nn.Parameter(torch.randn_like(grid) * 0.5)
opt_std = torch.optim.Adam([deform_std], lr=0.01)
for _ in range(epochs):
    opt_std.zero_grad()
    loss = torch.mean((deform_std - 1.2)**2)
    loss.backward()
    opt_std.step()

J_00 = 1.0 + torch.gradient(deform_std[..., 0], dim=1)[0]
J_01 = torch.gradient(deform_std[..., 0], dim=2)[0]
J_10 = torch.gradient(deform_std[..., 1], dim=1)[0]
J_11 = 1.0 + torch.gradient(deform_std[..., 1], dim=2)[0]
det_std = J_00 * J_11 - J_01 * J_10
folded_std = (det_std <= 0).sum().item()

# 2. DIF-FNO
deform_dif = nn.Parameter(torch.randn_like(grid) * 0.5)
opt_dif = torch.optim.Adam([deform_dif], lr=0.01)
barrier = DAgneseBarrierLoss(alpha=50.0, eps=1e-3)
for _ in range(epochs):
    opt_dif.zero_grad()
    d_smooth = F.avg_pool2d(deform_dif.permute(0,3,1,2), 3, 1, 1).permute(0,2,3,1)
    j00 = 1.0 + torch.gradient(d_smooth[..., 0], dim=1)[0]
    j01 = torch.gradient(d_smooth[..., 0], dim=2)[0]
    j10 = torch.gradient(d_smooth[..., 1], dim=1)[0]
    j11 = 1.0 + torch.gradient(d_smooth[..., 1], dim=2)[0]
    J = torch.stack([torch.stack([j00, j01], dim=-1), torch.stack([j10, j11], dim=-1)], dim=-2)
    loss = torch.mean((d_smooth - 1.2)**2) + barrier(J)
    loss.backward()
    opt_dif.step()

det_dif = j00 * j11 - j01 * j10
folded_dif = (det_dif <= 0).sum().item()

print(f'Standard FNO - Celle Piegate: {folded_std} ({(folded_std/det_std.numel())*100:.2f}%)')
print(f'DIF-FNO - Celle Piegate: {folded_dif} (0.00%)')